# Post-processing and standard plots

This tutorial introduces the standardized post-processing interface. We run a small Vlasov–Ampère example, load its output as an autocomplete-friendly `RunOutput`, and make the plots most commonly used to inspect a simulation.

For a production run you can skip the simulation setup and construct both objects with `path_out="path/to/sim"` instead.

In [ ]:
import os
import tempfile

from struphy import (
    BinningPlot,
    BoundaryParameters,
    DerhamOptions,
    EnvironmentOptions,
    LoadingParameters,
    SavingParameters,
    Simulation,
    SortingParameters,
    Time,
    WeightsParameters,
    domains,
    grids,
    maxwellians,
    perturbations,
)
from struphy.diagnostics.plotting import (
    GrowthFit,
    InteractiveSliceViewer,
    View,
    plot_marker_trajectories,
    plot_panels,
    plot_scalars,
    plot_slice,
    plot_timeseries,
)
from struphy.models import VlasovAmpereOneSpecies

## Create a compact demonstration run

Post-processing operates on a completed run. The small setup below saves an electric field, a few marker trajectories, scalar diagnostics, and a binned $(\eta_1,v_1)$ distribution. These are the main output types handled by the plotting interface.

In [ ]:
model = VlasovAmpereOneSpecies(alpha=1.0, epsilon=-1.0, with_B0=False)
model.em_fields.e_field.save_data = True
model.em_fields.phi.save_data = True
model.kinetic_ions.var.save_data = True

model.propagators.push_eta.options = model.propagators.push_eta.Options()
model.propagators.coupling_va.options = model.propagators.coupling_va.Options()
model.initial_poisson.options = model.initial_poisson.Options(stab_mat="M0")

binplot = BinningPlot(
    slice="e1_v1",
    n_bins=(32, 32),
    ranges=((0.0, 1.0), (-5.0, 5.0)),
)
model.kinetic_ions.set_markers(
    loading_params=LoadingParameters(ppc=32, seed=1234),
    weights_params=WeightsParameters(control_variate=True),
    boundary_params=BoundaryParameters(),
    sorting_params=SortingParameters(boxes_per_dim=(4, 1, 1), do_sort=True),
    saving_params=SavingParameters(n_markers=12, binning_plots=(binplot,)),
)

background = maxwellians.Maxwellian3D(n=(1.0, None))
model.kinetic_ions.var.add_background(background)
density_mode = perturbations.ModesCos(ls=(1,), amps=(1e-3,))
model.kinetic_ions.var.add_initial_condition(maxwellians.Maxwellian3D(n=(1.0, density_mode)))

In [ ]:
demo_tmp = tempfile.TemporaryDirectory(prefix="struphy_postprocessing_")
demo_root = demo_tmp.name

env = EnvironmentOptions(
    out_folders=demo_root,
    sim_folder="vlasov_ampere_demo",
    save_restart=False,
)
sim = Simulation(
    model=model,
    env=env,
    time_opts=Time(dt=0.1, Tend=0.4),
    domain=domains.Cuboid(r1=2 * 3.141592653589793),
    grid=grids.TensorProductGrid(num_elements=(8, 1, 1)),
    derham_opts=DerhamOptions(degree=(2, 1, 1)),
)
sim.run()
print(f"Raw output: {sim.env.path_out}")

## Process and load the output

`sim.pproc(load=True)` evaluates saved FEEC fields, organizes particle diagnostics, and returns a lazy `RunOutput`. `physical=True` additionally creates physical field components; `create_vtk=False` keeps this notebook quick. With `force=False`, a complete post-processing result is reused.

Individual products are standard `xarray.DataArray` objects with named dimensions, coordinates, units, and labels. Arrays are loaded only when accessed. `RunOutput.open(path_out)` can load an already processed run without running post-processing again.

In [ ]:
run = sim.pproc(
    physical=True,
    create_vtk=False,
    force=False,
    load=True,
)

Products are arranged into clear namespaces. VS Code and interactive shells can complete the available names after a run is opened: fields are grouped by field species, while distribution and density products are grouped by species and saved slice. Flat catalogs remain available for code that needs to iterate over arbitrary products.

In [ ]:
print("scalars:", tuple(run.scalars.data_vars))
print("field species:", tuple(run.fields))
print("distribution species:", tuple(run.distributions))
print("particle species:", tuple(run.orbits))
print("all field products:", tuple(run.field_catalog))

phase_space = run.distributions.kinetic_ions.e1_v1_density.f_binned
print(phase_space)
print("dimensions:", phase_space.dims)
print("time coordinate:", phase_space.t)

## Scalar overview and time series

`plot_scalars()` gives a quick overview of every recorded scalar. If `total_energy` is available, it can also be used for the conservation-error panel. Individual time series can be shown on linear or logarithmic axes, and `GrowthFit` restricts an exponential fit to a chosen time interval. Plot functions return an already-rendered `PlotResult`; calling `.save()` never draws a second figure.

In [ ]:
scalar_plot, energy_error = plot_scalars(
    run.scalars,
    error_panel="total_energy",
    run_label=run.label,
)
scalar_plot.fig

In [ ]:
electric_energy = run.scalars.electric_energy
energy_plot = plot_timeseries(
    electric_energy,
    logy=True,
    fit=GrowthFit(
        window=(0.0, 0.4 * run.units.t),
        amplitude_from_quadratic=True,
    ),
    title="Electric-field energy",
    run_label=run.label,
)
print("growth rate:", energy_plot.fit_results[0].rate)

## Two-dimensional data

Named xarray selection keeps plots readable. Use `.isel()` for an integer index and `.sel(..., method="nearest")` for the point nearest a coordinate value. A `View` records the display axes, coordinate system, and reusable selections.

In [ ]:
final_distribution = phase_space.isel(t=-1)
plot_slice(
    final_distribution,
    view=View(x="e1", y="v1"),
    equal_aspect=False,
    title="Final phase-space distribution",
    run_label=run.label,
).fig

For a compact view of the evolution, `plot_panels()` chooses evenly spaced snapshots. The same `View` can later drive an interactive viewer or animation. `shared_clim=True` makes panel colors directly comparable.

In [ ]:
phase_view = View(x="e1", y="v1")
plot_panels(
    phase_space,
    view=phase_view,
    nrows=1,
    ncols=5,
    shared_clim=True,
    title="Phase-space evolution",
    run_label=run.label,
).fig

## Interactive plots

`InteractiveSliceViewer` adds one slider for every dimension not assigned to the display axes. In JupyterLab, run `%matplotlib widget` before this cell if `ipympl` is installed; the default inline backend still displays the initial frame. Keep the viewer alive so its callbacks remain connected.

In [ ]:
phase_viewer = InteractiveSliceViewer(
    phase_space,
    view=phase_view,
    run_label=run.label,
)
phase_viewer.draw().fig

Saved marker orbits are grouped by species. `plot_marker_trajectories()` draws their three-dimensional paths, while `max_markers` limits rendering cost for large production runs.

In [ ]:
orbit_plot = plot_marker_trajectories(
    run.orbits.kinetic_ions,
    max_markers=12,
    show_paths=True,
)
orbit_plot.fig

## Save standard output

Every `PlotResult` supports `.save(path)`. For a complete scalar report, `run.save_scalar_plots()` writes a CSV table, an overview, and one PNG per scalar beneath `post_processing/scalars/`.

In [ ]:
written = run.save_scalar_plots()
print("Wrote:")
for path in written:
    print(" ", os.path.relpath(path, run.path_out))

## Apply the workflow to another run

For an already completed simulation, the complete loading pattern is:

```python
path_out = "/path/to/sim_1"
from struphy import RunOutput, post_process
run = post_process(path_out=path_out, physical=True, force=False)
# If it is already processed:
run = RunOutput.open(path_out)
```

Use `run.scalars`, `run.fields`, `run.distributions`, `run.orbits`, and `run.densities`. Attribute access is the normal interactive API; the corresponding `*_catalog` mappings are intended for generic loops and tooling.